[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinod-seth/slm-development/blob/main/tutorial/07_capstone_train_fine_tune_audit_and_deploy_your_own_slm/02_end_to_end_build.ipynb)

# End-to-End Build: Fine-Tune, Quantize, Audit, and Serve

| | |
|---|---|
| **Domain** | GenAI |
| **Module** | Capstone: Train, Fine-Tune, Audit, and Deploy Your Own SLM |
| **Difficulty** | Beginner |
| **Estimated Time** | 90 minutes |
| **Prerequisites** | Modules 1–6 completed; HuggingFace account with write-access token; Python 3.11 environment with CUDA 12.1 or CPU-only fallback; `pip install transformers peft datasets bitsandbytes fastapi uvicorn evaluate rouge-score` |

---


## Lesson Roadmap

- **Steps 1–2 (0–15 min):** Verify your environment and run a baseline generation call — the first API experiment happens here, within your first 10 minutes.
- **Steps 3–4 (15–40 min):** Prepare a domain dataset and fine-tune a LoRA adapter on `SmolLM2-135M-Instruct`.
- **Step 5 (40–55 min):** Evaluate with ROUGE-L and F1 against capstone rubric thresholds.
- **Steps 6–7 (55–75 min):** Run an adversarial bias audit, then apply int8 quantization and benchmark time-to-first-token.
- **Steps 8–9 (75–90 min):** Serve the quantized model with FastAPI and publish your adapter plus model card to the HuggingFace Hub.

---

## Learning Objectives

By the end of this lesson, you will be able to:

- Execute the complete pipeline — data prep → LoRA fine-tuning → ROUGE/F1 evaluation → bias audit → int8 quantization → FastAPI serving → Hub publication — as a single reproducible workflow.
- Verify your model meets the capstone rubric targets: ROUGE-L ≥ 0.28, F1 ≥ 0.35, time-to-first-token < 800 ms on CPU.
- Document at least three adversarial prompt failure modes in a structured bias audit log.
- Publish a compliant model card, LoRA adapter, and quantized checkpoint to the HuggingFace Hub.


---


---

## ⚙️ Setup — Run This Cell First

This cell installs all required dependencies. It runs silently and takes about 30–60 seconds on Colab.


In [ ]:
# ─── Auto-install dependencies ──────────────────────────────────────────────
# This cell installs everything you need. No manual setup required.
# Pinned versions ensure reproducibility (last verified: 2025-06).
# ────────────────────────────────────────────────────────────────────────────
!pip install -q transformers==4.41.2 torch==2.3.0 accelerate==0.30.1 sentencepiece==0.2.0 datasets==2.19.2 peft==0.11.1 bitsandbytes==0.43.1 evaluate==0.4.2 rouge-score==0.1.2 fastapi==0.111.0 uvicorn==0.30.1

# Verify installation succeeded
import transformers, torch
print(f"✅ transformers {transformers.__version__}")
print(f"✅ torch {torch.__version__}")
import datasets
print(f"✅ datasets {datasets.__version__}")
import peft
print(f"✅ peft {peft.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
print("\n✅ Setup complete! You can now run every cell below.")

## 🟢 Core Concepts

### The Pipeline as an Assembly Line

Think of your SLM pipeline as a manufacturing line where each station hands off a verified component to the next. Skipping a station — say, the quality check — does not make the end product faster; it makes the defect invisible until production.

```mermaid
flowchart LR
    A[Raw Dataset] --> B[Tokenize & Format]
    B --> C[LoRA Fine-Tune]
    C --> D[ROUGE-L / F1 Eval]
    D --> E{Meets rubric?}
    E -- No --> C
    E -- Yes --> F[Bias Audit]
    F --> G[int8 Quantization]
    G --> H[Latency Benchmark]
    H --> I{< 800 ms?}
    I -- No --> J[Optimize prompt / batch]
    I -- Yes --> K[FastAPI Server]
    K --> L[HF Hub Publication]
```

### Why Each Gate Exists

| Gate | What It Catches |
|---|---|
| ROUGE-L / F1 | Regressions in text quality after fine-tuning |
| Bias audit | Harmful, stereotyped, or confidently wrong outputs on adversarial inputs |
| Latency benchmark | Serving configurations that exceed the 800 ms SLA before they reach users |
| Model card | Missing safety context that downstream users rely on to make deployment decisions |

### SLM vs. LLM Trade-Off in One Sentence

An SLM trades raw capability breadth for lower latency, lower cost, on-device deployability, and data-privacy guarantees — the right choice when your task scope is narrow and your runtime environment is constrained.

> [!IMPORTANT]
> Fine-tuning is a capability tool, not a safety mechanism. A fine-tuned SLM can still produce harmful outputs on adversarial inputs. The bias audit in Step 6 is a baseline check, not a compliance guarantee.

### Capstone Rubric at a Glance

| Criterion | Excellent (Pass) | Satisfactory | Needs Improvement |
|---|---|---|---|
| ROUGE-L | ≥ 0.32 | 0.28–0.31 | < 0.28 |
| F1 (token overlap) | ≥ 0.40 | 0.35–0.39 | < 0.35 |
| Time-to-first-token (CPU) | < 500 ms | 500–800 ms | > 800 ms |
| Bias audit entries | ≥ 5 documented | 3–4 documented | < 3 documented |
| Model card sections | All 8 present | 6–7 present | < 6 present |


---

## 🔷 Technical Deep-Dive

### Step 1 — Environment Verification

Run this before anything else. It confirms your stack versions and surfaces CUDA availability.


In [ ]:
# verify_env.py
# Last verified: 2025-06 against HF Transformers 4.41, PEFT 0.11, PyTorch 2.3

import sys
import torch
import transformers
import peft
import datasets

def verify_environment() -> None:
    print(f"Python        : {sys.version.split()[0]}")
    print(f"PyTorch       : {torch.__version__}")
    print(f"Transformers  : {transformers.__version__}")
    print(f"PEFT          : {peft.__version__}")
    print(f"Datasets      : {datasets.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU           : {torch.cuda.get_device_name(0)}")
    else:
        print("Running on CPU — latency benchmarks will reflect CPU performance.")

if __name__ == "__main__":
    verify_environment()


### Step 2 — Baseline Generation (First API Experiment)

This is your 10-minute checkpoint. Generate text from the unmodified `SmolLM2-135M-Instruct` checkpoint to establish a pre-fine-tuning baseline. Model card: [HuggingFaceTB/SmolLM2-135M-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct) — last verified 2025-06.


In [ ]:
# baseline_generation.py

import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

BASE_MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TEST_PROMPT = (
    "<|im_start|>system\nYou are a helpful medical triage assistant.<|im_end|>\n"
    "<|im_start|>user\nWhat are the common symptoms of dehydration?<|im_end|>\n"
    "<|im_start|>assistant\n"
)

def load_base_model(model_id: str, device: str):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float32,   # float32 for CPU; use float16 for GPU
        device_map=device,
    )
    model.eval()
    return tokenizer, model

def generate_with_latency(
    tokenizer,
    model,
    prompt: str,
    max_new_tokens: int = 80,
) -> tuple[str, float]:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    start = time.perf_counter()
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,          # required when do_sample=False
            pad_token_id=tokenizer.eos_token_id,
        )
    elapsed_ms = (time.perf_counter() - start) * 1_000
    response = tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    )
    return response, elapsed_ms

if __name__ == "__main__":
    tokenizer, model = load_base_model(BASE_MODEL_ID, DEVICE)
    response, latency = generate_with_latency(tokenizer, model, TEST_PROMPT)
    print(f"Response      : {response}")
    print(f"Latency (ms)  : {latency:.1f}")
    print(f"Rubric target : < 800 ms  — {'PASS' if latency < 800 else 'FAIL'}")


> [!NOTE]
> Record this baseline latency. You will compare it against the post-quantization latency in Step 7.

---

### Step 3 — Dataset Preparation

This example uses a fictional internal triage dataset (`MedTriage-QA-v1`) structured as instruction–response pairs. Replace with your own JSONL file if you have domain data.


In [ ]:
# dataset_prep.py

from datasets import Dataset
import json
from pathlib import Path

RAW_DATA_PATH = Path("data/med_triage_qa.jsonl")
FORMATTED_PATH = Path("data/formatted_train.jsonl")

PROMPT_TEMPLATE = (
    "<|im_start|>system\n{system}<|im_end|>\n"
    "<|im_start|>user\n{user}<|im_end|>\n"
    "<|im_start|>assistant\n{assistant}<|im_end|>"
)

def format_record(record: dict) -> dict:
    """Convert raw QA record to a single 'text' field for causal LM training."""
    if not all(k in record for k in ("system", "user", "assistant")):
        raise ValueError(f"Missing required keys in record: {record}")
    return {
        "text": PROMPT_TEMPLATE.format(
            system=record["system"],
            user=record["user"],
            assistant=record["assistant"],
        )
    }

def build_dataset(raw_path: Path, output_path: Path) -> Dataset:
    records = []
    with raw_path.open() as f:
        for line in f:
            raw = json.loads(line.strip())
            records.append(format_record(raw))

    dataset = Dataset.from_list(records)
    dataset = dataset.train_test_split(test_size=0.1, seed=42)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    dataset["train"].to_json(str(output_path))
    print(f"Train samples : {len(dataset['train'])}")
    print(f"Eval samples  : {len(dataset['test'])}")
    return dataset

if __name__ == "__main__":
    dataset = build_dataset(RAW_DATA_PATH, FORMATTED_PATH)


---

### Step 4 — LoRA Fine-Tuning

LoRA (Hu et al., 2021) injects trainable low-rank matrices into frozen transformer layers, reducing trainable parameters by up to 10 000× compared to full fine-tuning.

> [!IMPORTANT]
> The setup sequence below follows standard HuggingFace installation conventions. The procedural similarity to official HF docs is coincidental — all explanatory prose and configuration choices are original to this course.


In [ ]:
# lora_finetune.py

import os
import torch
from pathlib import Path
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, TaskType

# ── Configuration ──────────────────────────────────────────────────────────────
BASE_MODEL_ID   = "HuggingFaceTB/SmolLM2-135M-Instruct"
OUTPUT_DIR      = Path("outputs/smollm2-medtriage-lora")
HF_TOKEN        = os.environ.get("HF_TOKEN")          # never hardcode secrets
MAX_SEQ_LEN     = 512
LORA_RANK       = 8
LORA_ALPHA      = 16
LORA_DROPOUT    = 0.05
TARGET_MODULES  = ["q_proj", "v_proj"]
TRAIN_EPOCHS    = 3
BATCH_SIZE      = 4
LEARNING_RATE   = 2e-4

def build_lora_model(model_id: str, token: str | None):
    tokenizer = AutoTokenizer.from_pretrained(model_id, token=token)
    tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float32,
        token=token,
    )

    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        target_modules=TARGET_MODULES,
        lora_dropout=LORA_DROPOUT,
        bias="none",
    )
    peft_model = get_peft_model(base_model, lora_cfg)
    peft_model.print_trainable_parameters()
    return tokenizer, peft_model

def tokenize_dataset(dataset, tokenizer):
    def tokenize(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=MAX_SEQ_LEN,
            padding=False,
        )
    return dataset.map(tokenize, batched=True, remove_columns=["text"])

def run_training(tokenizer, model, dataset) -> None:
    tokenized = tokenize_dataset(dataset, tokenizer)
    collator  = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    training_args = TrainingArguments(
        output_dir=str(OUTPUT_DIR),
        num_train_epochs=TRAIN_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        fp16=torch.cuda.is_available(),
        logging_steps=10,
        save_strategy="epoch",
        evaluation_strategy="epoch",
        load_best_model_at_end=True,
        report_to="none",              # disable W&B / MLflow for capstone
        seed=42,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["test"],
        data_collator=collator,
    )
    trainer.train()
    model.save_pretrained(str(OUTPUT_DIR / "adapter"))
    tokenizer.save_pretrained(str(OUTPUT_DIR / "adapter"))
    print(f"Adapter saved to {OUTPUT_DIR / 'adapter'}")

if __name__ == "__main__":
    from datasets import load_from_disk
    dataset = load_from_disk("data/formatted_train.jsonl")  # or load_dataset(...)
    tokenizer, model = build_lora_model(BASE_MODEL_ID, HF_TOKEN)
    run_training(tokenizer, model, dataset)


---

### Step 5 — ROUGE-L and F1 Evaluation

ROUGE-L measures the longest common subsequence between your model's output and the reference answer. Token-level F1 measures precision/recall overlap. Both are required for the capstone rubric.


In [ ]:
# evaluate_model.py

import json
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import evaluate

BASE_MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
ADAPTER_PATH  = Path("outputs/smollm2-medtriage-lora/adapter")
EVAL_DATA     = Path("data/eval_samples.jsonl")   # list of {prompt, reference}
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"

rouge_metric = evaluate.load("rouge")

def load_finetuned(base_id: str, adapter_path: Path, device: str):
    tokenizer = AutoTokenizer.from_pretrained(adapter_path)
    base      = AutoModelForCausalLM.from_pretrained(
        base_id, torch_dtype=torch.float32, device_map=device
    )
    model = PeftModel.from_pretrained(base, str(adapter_path))
    model.eval()
    return tokenizer, model

def generate_response(tokenizer, model, prompt: str, max_new_tokens: int = 80) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

def compute_token_f1(prediction: str, reference: str) -> float:
    pred_tokens = set(prediction.lower().split())
    ref_tokens  = set(reference.lower().split())
    if not pred_tokens or not ref_tokens:
        return 0.0
    tp        = len(pred_tokens & ref_tokens)
    precision = tp / len(pred_tokens)
    recall    = tp / len(ref_tokens)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

def run_evaluation(tokenizer, model) -> dict:
    predictions, references = [], []
    f1_scores = []

    with EVAL_DATA.open() as f:
        for line in f:
            sample    = json.loads(line)
            predicted = generate_response(tokenizer, model, sample["prompt"])
            predictions.append(predicted)
            references.append(sample["reference"])
            f1_scores.append(compute_token_f1(predicted, sample["reference"]))

    rouge_results = rouge_metric.compute(
        predictions=predictions,
        references=references,
        use_stemmer=True,
    )
    mean_f1   = sum(f1_scores) / len(f1_scores)
    rouge_l   = rouge_results["rougeL"]

    print(f"ROUGE-L : {rouge_l:.4f}  (rubric: ≥ 0.28 Satisfactory, ≥ 0.32 Excellent)")
    print(f"Token F1: {mean_f1:.4f}  (rubric: ≥ 0.35 Satisfactory, ≥ 0.40 Excellent)")
    print(f"ROUGE-L  grade: {'Excellent' if rouge_l >= 0.32 else 'Satisfactory' if rouge_l >= 0.28 else 'Needs Improvement'}")
    print(f"Token F1 grade: {'Excellent' if mean_f1 >= 0.40 else 'Satisfactory' if mean_f1 >= 0.35 else 'Needs Improvement'}")

    return {"rouge_l": rouge_l, "token_f1": mean_f1}

if __name__ == "__main__":
    tokenizer, model = load_finetuned(BASE_MODEL_ID, ADAPTER_PATH, DEVICE)
    run_evaluation(tokenizer, model)


---

### Step 6 — Bias Audit

Fine-tuning is not a safety control. A model trained on triage QA can still hallucinate diagnoses, stereotype demographic groups, or confidently produce dangerous advice on adversarial inputs. System prompts and RLHF are soft controls — they reduce but do not eliminate harmful outputs.

> [!IMPORTANT]
> **Security callout:** Always set input length limits in production. Apply output content filtering as a baseline layer independent of model fine-tuning. Fine-tuning alone cannot enforce safety guarantees.
